### Multiple Linear Regression with `Carseats` Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from ISLP import load_data

Carseats = load_data('Carseats')
Carseats.columns

In [ ]:
Carseats

In [ ]:
pd.plotting.scatter_matrix(Carseats, figsize=(10,10));

#### (a) Fit a Multiple Regression Model with to predict `Sales` with `Price`, `Urban` and `US`

- Price is quantitative
- Urban and US are both qualitative variables so I will need to one-hot encode them

Some notes
- F-statistic probability looks good, shows that there is a relationship between at least one variable.
- t-statistic probabilities for each variable looks good except for `Urban`. Perhaps `Urban` does not have a relationship with `Sales` in the presence of `Price` and `US`.

In [ ]:
y = Carseats[['Sales']]
X = pd.DataFrame({
    'Intercept': np.ones(Carseats.shape[0]),
    'Price': Carseats['Price'],
    'Urban': Carseats['Urban'],
    'US': Carseats['US'],
})
X = pd.get_dummies(X, columns=['Urban', 'US'], drop_first=True, dtype=int)

In [ ]:
results = sm.OLS(y, X).fit()
results.summary()

#### (b) Provide an interpretation of each coefficient in the model

Coefficients:
- Intercept: Value of `Sales` when all variables `Price`, `Urban`, `US` are zero.
- `Price`: Increase in `Sales` for each unit of `Price`
- `Urban`: Increase in `Sales` when the store is in an Urban location
- `US`: Increase in `Sales` when the store is in the US

#### (c) Model in equation form

$\hat{Y} = \hat{\beta}_0 + \hat{\beta}_1x_1 + \hat{\beta}_2x_2 + \hat{\beta}_3x_3$

Where

$x_1$ is the Price

$x_2$ is 1 if the store is in an Urban location, 0 otherwise

$x_3$ is 1 if the store is in the US, 0 otherwise

#### (d) Predicts that we can reject the null hypothesis.

1. In the absence of `Price`, `Urban`, and `US`, we can reject the null hypothesis that the intercept is 0.
2. In the presence of `Urban` and `US`, we can reject the null hypothesis that `Price` has no relationship with `Sales`.
3. In the presence of `Price` and `Urban`, we can reject the null hypothesis that `US` has no relationship with `Sales`.

#### (e) Fit smaller model with only predictors that we rejected the null hypothesis for

Notes
- I removed `Urban` from the last model
- It seems that R^2 did not change, and the coefficients for the other variables did not change much.

In [ ]:
X_small = pd.DataFrame({
    'Intercept': np.ones(Carseats.shape[0]),
    'Price': Carseats['Price'],
    'US': Carseats['US']
})
X_small = pd.get_dummies(X_small, columns=['US'], drop_first=True, dtype=int)
X_small[:4]

In [ ]:
results_small = sm.OLS(y, X_small).fit()
results_small.summary()

#### (f) How well do the models in (a) and (e) fit the data?

- R^2 value is 0.239 for both models, showing that they explain 0.239 of the variance in the model.
- RSE value is about 6.09 for (e) and 6.11 for (a). This is quite a large proportion of the mean of the fitted values. This shows that the model does not do very much for explaining the relationship and is not much better than a random guess.
- Residual plots show no discernible pattern. We can conclude that based on `Price` and `US` variables, the data is somewhat linear and can be modelled by a linear model.


In [ ]:
print(f"RSE for (a) is {results.scale}, this is {results.scale / np.mean(results.fittedvalues)} of the mean of fitted values")
print(f"RSE for (e) is {results_small.scale}, this is {results_small.scale / np.mean(results_small.fittedvalues)} of the mean of fitted values")

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(12,6))
axs[0].scatter(x=results.fittedvalues, y=results.resid)
axs[0].set_xlabel("fitted values")
axs[0].set_ylabel("residual")
axs[0].axhline(y=0, c='k', ls='--')

axs[1].scatter(x=results_small.fittedvalues, y=results_small.resid)
axs[1].set_xlabel("fitted values")
axs[1].set_ylabel("residual")
axs[1].axhline(y=0, c='k', ls='--')

plt.tight_layout()

#### (g) 95% confidence intervals for model in (e)
- Intercept: [11.790, 14.271]
- Price: [-0.065, -0.044]
- US: [0.692, 1.708]

#### (h) Outlier and Leverage Analysis for model in (e)
- We will plot a studentized residual plot for outlier analysis, and a leverage plot for leverage analysis
- Outlier analysis: All points are within 3 standard errors so there are no outliers
- Leverage: One point near 50 has higher leverage than the rest, but since there are no outliers this should be fine.

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(12,6))
axs[0].scatter(x=results_small.fittedvalues, y=results_small.get_influence().resid_studentized_internal)
axs[1].scatter(x=X_small.index, y=results_small.get_influence().hat_matrix_diag)